# Rainfall Classification: External Validation

This notebook evaluates pipelines and LdaBoost on a held-out test split, with reproducible settings and relative imports.


In [6]:
# Reproducible setup and imports
import os
import random
import logging

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split

# Global seed
RANDOM_SEED: int = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Import LdaBoost via relative path
import sys
REL_PROJECT_ROOT = "../../"
if REL_PROJECT_ROOT not in sys.path:
    sys.path.insert(0, REL_PROJECT_ROOT)
from LdaBoost import LdaBoost


In [7]:
N_ESTIMATOR = 100
LEARNING_RATE = 0.1
MAX_DEPTH = 3

## Dataset and preprocessing


In [8]:
import re

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in df.columns]
    return df

train_df = pd.read_csv("Data/train.csv", index_col="id")
train_df = clean_column_names(train_df)

y = train_df["rainfall"]
X = train_df.drop(columns="rainfall").copy()

le = LabelEncoder()
y_enc = le.fit_transform(y)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=RANDOM_SEED, stratify=y_enc
)

features = list(X.columns)



## Baselines on external test: GBM, PCA+GBM, LDA+GBM


In [9]:
# Common scaler
scaler = Pipeline([("scaler", StandardScaler())])

# 1) GBM baseline
base_preprocess = ColumnTransformer(
    remainder="passthrough",
    transformers=[("scale", scaler, features)],
)
base_gbm = Pipeline([
    ("preprocess", base_preprocess),
    ("gb", GradientBoostingClassifier(n_estimators=N_ESTIMATOR, learning_rate=LEARNING_RATE, max_depth=MAX_DEPTH, random_state=RANDOM_SEED)),
])
base_gbm.fit(X_train, y_train)
y_pred = base_gbm.predict(X_test)
print(f"GBM — Test accuracy: {accuracy_score(y_test, y_pred):.4f}")

# 2) PCA + GBM
pca_preprocess = ColumnTransformer(
    remainder="drop",
    transformers=[
        (
            "pca",
            Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=max(1, X.shape[1] // 2), random_state=RANDOM_SEED)),
            ]),
            features,
        )
    ],
)
pca_gbm = Pipeline([
    ("preprocess", pca_preprocess),
    ("gb", GradientBoostingClassifier(n_estimators=N_ESTIMATOR, learning_rate=LEARNING_RATE, max_depth=MAX_DEPTH, random_state=RANDOM_SEED)),
])
pca_gbm.fit(X_train, y_train)
y_pred = pca_gbm.predict(X_test)
print(f"PCA+GBM — Test accuracy: {accuracy_score(y_test, y_pred):.4f}")

# 3) LDA + GBM
lda_preprocess = ColumnTransformer(
    remainder="drop",
    transformers=[
        (
            "lda",
            Pipeline([
                ("scaler", StandardScaler()),
                ("lda", LDA(n_components=None)),
            ]),
            features,
        )
    ],
)
lda_gbm = Pipeline([
    ("preprocess", lda_preprocess),
    ("gb", GradientBoostingClassifier(n_estimators=N_ESTIMATOR, learning_rate=LEARNING_RATE, max_depth=MAX_DEPTH, random_state=RANDOM_SEED)),
])
lda_gbm.fit(X_train, y_train)
y_pred = lda_gbm.predict(X_test)
print(f"LDA+GBM — Test accuracy: {accuracy_score(y_test, y_pred):.4f}")


GBM — Test accuracy: 0.8676
PCA+GBM — Test accuracy: 0.8493
LDA+GBM — Test accuracy: 0.8379


## LdaBoost on external test


In [10]:
X_train_np = X_train.to_numpy()
X_test_np = X_test.to_numpy()

model = LdaBoost(n_estimators=N_ESTIMATOR, learning_rate=LEARNING_RATE, max_depth=MAX_DEPTH, random_state=RANDOM_SEED)
model.fit(X_train_np, y_train)
y_pred = model.predict(X_test_np)

print(f"LdaBoost — Test accuracy: {accuracy_score(y_test, y_pred):.4f}")


LdaBoost — Test accuracy: 0.8516
